# Prototipo de Visión por Computador para Renovación Urbana en Bogotá
¿Qué potencial de renovación urbana tiene un barrio? Responder esa pregunta exige, primero, conocer el estado de sus edificaciones. En Bogotá, ese diagnóstico suele hacerse mediante inspecciones en campo: costosas, lentas y difíciles de escalar. En este proyecto propongo una alternativa: usar visión por computador para estimar el deterioro de las fachadas a partir de imágenes a nivel de calle.

## Como iniciamos

1. Creamos una carpeta en Google Drive.
2. Subir los archivos generados en Visual Studio: `2_evaluate_facades.py`, `3_app_visor.py`, `requirements.txt`, la carpeta `banco_img/` (73 fotos) y `manifiesto_banco.csv`.
3. En Colab se activa la GPU: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**.

In [ ]:
# 1) Montar Google Drive y ubicarse en la carpeta del proyecto
from google.colab import drive
drive.mount('/content/drive')

import os
RUTA = '/content/drive/MyDrive/TU_CARPETA'  # <-- ajusta a tu carpeta
os.chdir(RUTA)
print('Directorio de trabajo:', os.getcwd())
!ls -la

In [ ]:
# 2) Instalar dependencias (torch ya viene con GPU en Colab)
!pip install -q requests transformers safetensors accelerate huggingface_hub pillow numpy pandas folium streamlit streamlit-folium
print('Dependencias listas')

In [ ]:
# 3) Verificar GPU
import torch
print('torch', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Sin GPU. Ve a Entorno de ejecución -> Cambiar tipo -> GPU T4 y vuelve a ejecutar.')

In [ ]:
# 4) Evaluar las fachadas del banco local con CLIP
# Genera diagnostico_facades.csv con latitud, longitud, categoria, score y ruta_imagen.
!python 2_evaluate_facades.py --images banco_img --manifest manifiesto_banco.csv --model openai/clip-vit-large-patch14

## Ver el resultado

**Opción A:** descarga `resultado.zip` y abre el visor en tu PC no se requere GPU.

**Opción B:** abre el visor directamente en Colab con un túnel (celda siguiente).

In [ ]:
# 5) Empaquetar y descargar resultados
!zip -r resultado.zip banco_img diagnostico_facades.csv
from google.colab import files
files.download('resultado.zip')
print('En tu PC: descomprime y ejecuta  streamlit run 3_app_visor.py')

## Opción B: visor en Colab con túnel ngrok

Se crea el token en https://dashboard.ngrok.com/auth.


In [ ]:
# 6) Lanzar el visor Streamlit y exponerlo con ngrok
!pip install -q pyngrok
from pyngrok import ngrok
ngrok.set_auth_token(getpass.getpass('Pega tu token de ngrok: '))

import os, subprocess, time, getpass
os.environ['STREAMLIT_SERVER_PORT'] = '8501'
proc = subprocess.Popen(['streamlit', 'run', '3_app_visor.py', '--server.port', '8501', '--server.headless', 'true', '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false'])
time.sleep(8)
url = ngrok.connect(8501, 'http')
print('Abre el visor en:', url.public_url)